In [1]:
import sys

!{sys.executable} -m pip install -U openai


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import sys

!{sys.executable} -m pip install -U typing_extensions pydantic pydantic-core openai

  Using cached pydantic_core-2.47.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.5 kB)

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
# 설정 (RunPod)
import os, json, time
from collections import Counter, defaultdict
from openai import OpenAI

# OpenAI 키: 환경변수 우선, 없으면 입력 (코드에 하드코딩 금지)
oai = OpenAI(api_key=os.environ.get("OPENAI_API_KEY") or __import__("getpass").getpass("OPENAI_API_KEY: "))
GPT4O_MODEL = "gpt-4o"
O3_MODEL    = "o3"

# 우리 8B: base + LoRA 어댑터
BASE      = "kakaocorp/kanana-1.5-8b-instruct-2505"
OURS_PATH = "/workspace/kanana_finetuned_200"   # ← RunPod 내 어댑터 폴더

OPENAI_API_KEY:  ········


In [6]:
import os
for f in os.listdir(OURS_PATH):
    size = os.path.getsize(os.path.join(OURS_PATH, f)) / 1e6
    print(f"{f}: {size:.0f} MB")
# adapter_model.safetensors(~168MB), adapter_config.json 있어야 함

adapter_model.safetensors: 168 MB
tokenizer.json: 17 MB
tokenizer_config.json: 0 MB
training_args.bin: 0 MB
adapter_config.json: 0 MB
README.md: 0 MB
special_tokens_map.json: 0 MB


In [7]:
TASKS = {
    "CON": {"file":"CON_test_20_v2.json", "gold_field":"label",
            "labels":["일치","불일치","검토"], "input_fields":["clause_text","refs"]},
    "PEP": {"file":"PEP_test_20_v2.json", "gold_field":"output.label",
            "labels":["충족","불가","검토"], "input_fields":["input.criteria","input.rfp_excerpt","input.pep_excerpt"]},
    "RPT": {"file":"RPT_test_20_v2.json", "gold_field":"output.label",
            "labels":["충족","불가","검토"], "input_fields":["input.criteria","input.pep_excerpt","input.rpt_excerpt"]},
}

def get_nested(d, path):
    cur = d
    for k in path.split('.'):
        cur = cur.get(k) if isinstance(cur, dict) else None
    return cur

def load_test(task):
    cfg = TASKS[task]
    raw = json.load(open(cfg["file"], encoding="utf-8"))
    data = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
    gold = {r["id"]: (get_nested(r, cfg["gold_field"]) if "." in cfg["gold_field"] else r.get(cfg["gold_field"])) for r in data}
    return data, gold, cfg

In [8]:
def load_prompt(path):
    with open(path, encoding='utf-8') as f:
        return f.read().strip()

JUDGE_PROMPT = {
    "CON": load_prompt("CON_system_prompt_bal.txt"),
    "PEP": load_prompt("PEP_system_prompt_bal.txt"),
    "RPT": load_prompt("RPT_system_prompt_bal.txt"),
}

def build_input(rec, cfg):
    item = {"id": rec["id"]}
    for f in cfg["input_fields"]:
        item[f.split(".")[-1]] = get_nested(rec, f) if "." in f else rec.get(f, "")
    return json.dumps(item, ensure_ascii=False)

In [9]:
USAGE = {"GPT-4o": {"in":0,"out":0,"calls":0},
         "o3":     {"in":0,"out":0,"calls":0}}

def parse_label(txt, labels):
    if not txt: return None
    try:
        p = json.loads(txt)
        for key in ("label","판정","종합판정","재판정"):
            v = p.get(key) if isinstance(p, dict) else None
            if v in labels: return v
    except: pass
    for L in labels:
        if L in txt: return L
    return None

def predict_gpt4o(rec, cfg, task):
    for _ in range(3):
        try:
            r = oai.chat.completions.create(
                model=GPT4O_MODEL,
                messages=[{"role":"system","content":JUDGE_PROMPT[task]},
                          {"role":"user","content":build_input(rec, cfg)}],
                response_format={"type":"json_object"}, temperature=0)
            u = r.usage
            USAGE["GPT-4o"]["in"]  += u.prompt_tokens
            USAGE["GPT-4o"]["out"] += u.completion_tokens
            USAGE["GPT-4o"]["calls"] += 1
            return parse_label(r.choices[0].message.content, cfg["labels"])
        except Exception as e:
            print("  [gpt4o]", e); time.sleep(3)
    return None

def predict_o3(rec, cfg, task):
    for _ in range(3):
        try:
            r = oai.chat.completions.create(
                model=O3_MODEL,
                messages=[{"role":"system","content":JUDGE_PROMPT[task]},
                          {"role":"user","content":build_input(rec, cfg)}],
                response_format={"type":"json_object"}, max_completion_tokens=4000)
            u = r.usage
            USAGE["o3"]["in"]  += u.prompt_tokens
            USAGE["o3"]["out"] += u.completion_tokens   # reasoning 토큰 포함
            USAGE["o3"]["calls"] += 1
            return parse_label(r.choices[0].message.content, cfg["labels"])
        except Exception as e:
            print("  [o3]", e); time.sleep(3)
    return None

# ── 우리 8B ──
_ours = {"tok": None, "model": None}

def predict_ours(rec, cfg, task):
    import torch
    tok, model = _ours["tok"], _ours["model"]
    msgs = [{"role":"system","content":JUDGE_PROMPT[task]},
            {"role":"user","content":build_input(rec, cfg)}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=512, do_sample=False)
    txt = tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
    return parse_label(txt, cfg["labels"])

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

_ours = {}

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.bfloat16,   # 여기 dtype 말고 torch_dtype
)

_ours["model"] = PeftModel.from_pretrained(base, OURS_PATH).eval()
_ours["tok"] = AutoTokenizer.from_pretrained(OURS_PATH)

print("우리 8B 로드 완료 (base 4bit + LoRA 어댑터)")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

우리 8B 로드 완료 (base 4bit + LoRA 어댑터)


In [11]:
def compute_metrics(gold, pred, labels):
    ids = [i for i in gold if i in pred and pred[i] in labels]
    n = len(ids)
    acc = sum(1 for i in ids if gold[i]==pred[i]) / n if n else 0
    per, f1s = {}, []
    for L in labels:
        tp = sum(1 for i in ids if gold[i]==L and pred[i]==L)
        fp = sum(1 for i in ids if gold[i]!=L and pred[i]==L)
        fn = sum(1 for i in ids if gold[i]==L and pred[i]!=L)
        prec = tp/(tp+fp) if tp+fp else 0
        rec_ = tp/(tp+fn) if tp+fn else 0
        f1 = 2*prec*rec_/(prec+rec_) if prec+rec_ else 0
        per[L] = {"P":prec,"R":rec_,"F1":f1}; f1s.append(f1)
    macro_f1 = sum(f1s)/len(f1s)
    conf = {g:{p:0 for p in labels} for g in labels}
    for i in ids: conf[gold[i]][pred[i]] += 1
    return {"n":n,"acc":acc,"macro_f1":macro_f1,"per":per,"conf":conf}

def print_metrics(name, m, labels):
    print(f"\n[{name}] n={m['n']} acc={m['acc']:.3f} macro-F1={m['macro_f1']:.3f}")
    for L in labels:
        p = m["per"][L]
        print(f"  {L}: P={p['P']:.2f} R={p['R']:.2f} F1={p['F1']:.2f}")

In [13]:
from tqdm import tqdm

def normalize_gold_dict(gold):
    fixed = {}

    for k, v in gold.items():
        if isinstance(v, list):
            fixed[k] = v[0] if len(v) > 0 else "UNKNOWN"
        else:
            fixed[k] = v

    return fixed


ALL = {}

for task in ["CON","PEP","RPT"]:
    data, gold, cfg = load_test(task)

    # =========================
    # RPT gold가 ['검토']처럼 list로 들어오는 문제 보정
    # compute_metrics() 호출 전에 반드시 실행
    # =========================
    gold = normalize_gold_dict(gold)

    print(f"\n{'='*60}\n[{task}] test {len(data)}건\n{'='*60}")

    preds = {"GPT-4o": {}, "o3": {}, "Ours-8B": {}}
    # preds = {"Ours-8B": {}}

    for rec in tqdm(data, desc=task):
        preds["GPT-4o"][rec["id"]]  = predict_gpt4o(rec, cfg, task)   # 필요없으면 주석
        preds["o3"][rec["id"]]      = predict_o3(rec, cfg, task)      # 필요없으면 주석
        preds["Ours-8B"][rec["id"]] = predict_ours(rec, cfg, task)

    ALL[task] = {}

    for name, pred in preds.items():
        if not any(pred.values()):
            continue   # 빈(주석처리) 모델 스킵

        m = compute_metrics(gold, pred, cfg["labels"])
        print_metrics(f"{task}-{name}", m, cfg["labels"])
        ALL[task][name] = m


[CON] test 20건


CON: 100%|██████████| 20/20 [05:08<00:00, 15.40s/it]



[CON-GPT-4o] n=20 acc=0.900 macro-F1=0.896
  일치: P=0.89 R=1.00 F1=0.94
  불일치: P=0.86 R=0.86 F1=0.86
  검토: P=1.00 R=0.80 F1=0.89

[CON-o3] n=20 acc=0.950 macro-F1=0.955
  일치: P=0.89 R=1.00 F1=0.94
  불일치: P=1.00 R=0.86 F1=0.92
  검토: P=1.00 R=1.00 F1=1.00

[CON-Ours-8B] n=20 acc=0.600 macro-F1=0.579
  일치: P=0.54 R=0.88 F1=0.67
  불일치: P=0.60 R=0.43 F1=0.50
  검토: P=1.00 R=0.40 F1=0.57

[PEP] test 20건


PEP: 100%|██████████| 20/20 [06:23<00:00, 19.17s/it]



[PEP-GPT-4o] n=20 acc=0.850 macro-F1=0.796
  충족: P=0.78 R=1.00 F1=0.88
  불가: P=1.00 R=0.89 F1=0.94
  검토: P=0.67 R=0.50 F1=0.57

[PEP-o3] n=20 acc=0.750 macro-F1=0.551
  충족: P=0.60 R=0.86 F1=0.71
  불가: P=0.90 R=1.00 F1=0.95
  검토: P=0.00 R=0.00 F1=0.00

[PEP-Ours-8B] n=20 acc=0.750 macro-F1=0.555
  충족: P=0.70 R=1.00 F1=0.82
  불가: P=0.80 R=0.89 F1=0.84
  검토: P=0.00 R=0.00 F1=0.00

[RPT] test 20건


RPT: 100%|██████████| 20/20 [11:15<00:00, 33.78s/it]


[RPT-GPT-4o] n=20 acc=0.750 macro-F1=0.701
  충족: P=0.71 R=0.83 F1=0.77
  불가: P=0.89 R=0.89 F1=0.89
  검토: P=0.50 R=0.40 F1=0.44

[RPT-o3] n=20 acc=0.800 macro-F1=0.766
  충족: P=1.00 R=0.83 F1=0.91
  불가: P=0.69 R=1.00 F1=0.82
  검토: P=1.00 R=0.40 F1=0.57

[RPT-Ours-8B] n=20 acc=0.550 macro-F1=0.524
  충족: P=0.50 R=1.00 F1=0.67
  불가: P=0.75 R=0.33 F1=0.46
  검토: P=0.50 R=0.40 F1=0.44


In [14]:
print("ALL 존재 여부 확인")

try:
    print("ALL 타입:", type(ALL))
    print("ALL keys:", list(ALL.keys()))

    for task in ALL:
        print(task, "=>", list(ALL[task].keys()))
except NameError:
    print("ALL 변수가 아직 없음. 평가 셀을 먼저 다시 실행해야 함.")

ALL 존재 여부 확인
ALL 타입: <class 'dict'>
ALL keys: ['CON', 'PEP', 'RPT']
CON => ['GPT-4o', 'o3', 'Ours-8B']
PEP => ['GPT-4o', 'o3', 'Ours-8B']
RPT => ['GPT-4o', 'o3', 'Ours-8B']


In [15]:
import json
from tqdm import tqdm

ALL = {}
PRED_ALL = {}
GOLD_ALL = {}

for task in ["CON", "PEP", "RPT"]:
    data, gold, cfg = load_test(task)
    gold = normalize_gold_dict(gold)

    print(f"\n{'='*60}\n[{task}] test {len(data)}건\n{'='*60}")

    preds = {"GPT-4o": {}, "o3": {}, "Ours-8B": {}}

    for rec in tqdm(data, desc=task):
        rid = rec["id"]

        # 필요한 모델만 주석 해제
        # preds["GPT-4o"][rid] = predict_gpt4o(rec, cfg, task)
        # preds["o3"][rid] = predict_o3(rec, cfg, task)
        preds["Ours-8B"][rid] = predict_ours(rec, cfg, task)

    GOLD_ALL[task] = gold
    PRED_ALL[task] = preds

    ALL[task] = {}

    for name, pred in preds.items():
        if not any(pred.values()):
            continue

        m = compute_metrics(gold, pred, cfg["labels"])
        print_metrics(f"{task}-{name}", m, cfg["labels"])
        ALL[task][name] = m

# =========================
# 예측값/정답/결과 저장
# =========================

with open("eval_gold_all.json", "w", encoding="utf-8") as f:
    json.dump(GOLD_ALL, f, ensure_ascii=False, indent=2)

with open("eval_pred_all.json", "w", encoding="utf-8") as f:
    json.dump(PRED_ALL, f, ensure_ascii=False, indent=2)

with open("eval_metrics_all.json", "w", encoding="utf-8") as f:
    json.dump(ALL, f, ensure_ascii=False, indent=2)

print("저장 완료:")
print("- eval_gold_all.json")
print("- eval_pred_all.json")
print("- eval_metrics_all.json")


[CON] test 20건


CON: 100%|██████████| 20/20 [03:10<00:00,  9.52s/it]



[CON-Ours-8B] n=20 acc=0.600 macro-F1=0.579
  일치: P=0.54 R=0.88 F1=0.67
  불일치: P=0.60 R=0.43 F1=0.50
  검토: P=1.00 R=0.40 F1=0.57

[PEP] test 20건


PEP: 100%|██████████| 20/20 [03:59<00:00, 11.99s/it]



[PEP-Ours-8B] n=20 acc=0.750 macro-F1=0.555
  충족: P=0.70 R=1.00 F1=0.82
  불가: P=0.80 R=0.89 F1=0.84
  검토: P=0.00 R=0.00 F1=0.00

[RPT] test 20건


RPT: 100%|██████████| 20/20 [07:47<00:00, 23.40s/it]


[RPT-Ours-8B] n=20 acc=0.550 macro-F1=0.524
  충족: P=0.50 R=1.00 F1=0.67
  불가: P=0.75 R=0.33 F1=0.46
  검토: P=0.50 R=0.40 F1=0.44
저장 완료:
- eval_gold_all.json
- eval_pred_all.json
- eval_metrics_all.json


In [ ]:
def print_confusion_from_ALL(task, model_name):
    print(f"\n요청: {task}-{model_name}")

    m = ALL[task][model_name]
    conf = m["conf"]

    row_labels = list(conf.keys())

    col_labels = []
    for g in row_labels:
        for p in conf[g].keys():
            if p not in col_labels:
                col_labels.append(p)

    print(f"[{task}-{model_name}] Confusion Matrix")
    print("정답 \\ 예측".ljust(12), end="")

    for p in col_labels:
        print(str(p).rjust(8), end="")
    print()

    print("-" * (12 + 8 * len(col_labels)))

    for g in row_labels:
        print(str(g).ljust(12), end="")
        for p in col_labels:
            print(str(conf[g].get(p, 0)).rjust(8), end="")
        print()


print_confusion_from_ALL("CON", "Ours-8B")
print_confusion_from_ALL("PEP", "Ours-8B")
print_confusion_from_ALL("RPT", "Ours-8B")

In [ ]:
# PEP 검토 오답 확인용
task = "PEP"
data, gold, cfg = load_test(task)
gold = normalize_gold_dict(gold)

pred = {}
for rec in data:
    pred[rec["id"]] = predict_ours(rec, cfg, task)

print("PEP 정답=검토인데 틀린 케이스")
for rec in data:
    rid = rec["id"]
    if gold[rid] == "검토" and pred[rid] != "검토":
        print("=" * 80)
        print("id:", rid)
        print("gold:", gold[rid])
        print("pred:", pred[rid])
        print("\ninput:")
        print(rec.get("input", rec))